# RoundRobinGroupChat
RoundRobinGroupChat is a simple yet effective team configuration where all agents share the same context and take turns responding in a round-robin fashion. When it's their turn, each agent broadcasts their reply to all other agents, maintaining a consistent context across the entire team.

In [ ]:
#!pip install -U agent-framework

In [ ]:
import logging  
from typing import Any, List  
import os  
from dotenv import load_dotenv  
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

load_dotenv(override=True)

## Setting Up Tracer with OpenTelemetry
Using an OpenTelemetry tracer is convenient for debugging multi-agent systems.
Since `setup_observability` is not provided in this environment's Agent Framework,
we manually set up the OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector on port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Configure via environment variables (Agent Framework reads standard OTEL_* variables)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Disable Metrics (change as needed)

# Enable sensitive data collection (OpenAI IN/OUT data) by specifying enable_sensitive_data=True
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## Tracing and Monitoring AI Agents with Microsoft Foundry

In [ ]:
# # Install packages
# #pip install azure-ai-projects azure-identity azure-monitor-opentelemetry opentelemetry-sdk
# os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true" # False by default

# PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]

# # Code example
# from azure.ai.projects import AIProjectClient
# from azure.identity import DefaultAzureCredential
# from azure.monitor.opentelemetry import configure_azure_monitor

# # Create project client
# project_client = AIProjectClient(
#     # credential=DefaultAzureCredential(),
#     credential=DefaultAzureCredential(),
#     endpoint=os.environ["PROJECT_ENDPOINT"],
# )

# # Get Application Insights connection string and enable tracing
# connection_string = project_client.telemetry.get_application_insights_connection_string()
# configure_azure_monitor(connection_string=connection_string)

# # # Get tracer
# # from opentelemetry import trace
# # tracer = trace.get_tracer(__name__)

In [ ]:
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("\U0001f511 Authentication method: API Key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework auto-reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over credential, so explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("\U0001f510 Authentication method: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Tools Definition (MCP Streamable HTTP)

In [ ]:
# Create MCP Streamable HTTP tool (set longer timeout as initial connection may be slow)
mcp_tool = MCPStreamableHTTPTool(
    name="mcp_tools",
    url=mcp_server_uri,
    description="Tools for retrieving game shop product, order, inventory, user information, and Twitter analytics data.",
    timeout=120,
)


# Create Azure OpenAI client
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

print(f"MCP Tool: {mcp_tool}")

## Manager/Coordinator Agent
Set up a manager/coordinator agent for group chat orchestration.
The manager coordinates participants by selecting the next speaker based on the conversation state and task requirements. The manager is a full workflow participant with access to all agent infrastructure (tools, context, observability).
The manager agent must produce structured output compatible with ManagerSelectionResponse to communicate speaker selection decisions. Use response_format for reliable parsing. GroupChatBuilder

## Agent Definitions

In [ ]:
# Create coordinator agent with structured output for speaker selection
# Note: When using an Agent as an orchestrator, the structured output for speaker selection
# is set internally by GroupChat when invoking the agent (users don't need to set response_format).
coordinator = Agent(
    name="Coordinator",
    description="Selects speakers and coordinates multi-agent collaboration",
    instructions="""
You coordinate a team conversation to solve the user's task.

Review the conversation history and select the next speaker.

Guidelines:
- First, have the Researcher gather information
- Then have the Writer compose the final response
- Only terminate after both have made meaningful contributions
- Allow multiple rounds of information gathering if needed
""",
    client=chat_client,
 )

researcher = Agent(
    name="Researcher",
    description="Gathers relevant background information",
    instructions="Collect concise facts that help your teammates answer the question.",
    tools=[mcp_tool],
    client=chat_client,
 )

writer = Agent(
    name="Writer",
    description="Synthesizes polished responses from gathered information",
    instructions="Using the provided notes, create a clear and well-structured response.",
    client=chat_client,
 )

## Termination Conditions
In the current Agent Framework, termination is not done by returning `None` from the speaker selector (selection_func).
Termination is controlled by one of the following:
- `.with_termination_condition(...)` (determines termination based on conversation history)
- `.with_max_rounds(...)` (maximum round limit)

In [ ]:
workflow = (
    GroupChatBuilder(
        participants=[researcher, writer],
        orchestrator_agent=coordinator,
        # Termination condition: End conversation when 4 or more ASSISTANT role (agent) messages accumulate
        # This means termination occurs when Researcher and Writer have each responded once (orchestrator may also trigger early termination)
        termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4,
        # Required to pass participant agent streaming events to the consumer
        # When False (default), only the orchestrator's output is yielded,
        # and participant AgentResponseUpdate (token chunks) are filtered out
        intermediate_outputs=True,
    )
    .build()
 )

### What This Workflow Example Verifies (Coordinator / Researcher / Writer)

This example is a smoke test to verify that "multiple agents dividing tasks" can minimally function under orchestrator (Coordinator) leadership. It mainly verifies the following.

- **Whether orchestration works**: With `.with_orchestrator(agent=coordinator)`, the Coordinator can determine who (Researcher/Writer) to activate next
- **Whether role-based task division with tool usage works**: Only the Researcher has `tools=[mcp_tool]`, verifying that information retrieval (tool calls) → Writer integration division of labor works
- **Shared context maintenance**: Researcher's output remains in conversation history, and Writer can reference it to compose the final response
- **Whether termination control works**: With `with_termination_condition(...)`, the conversation stops when a specified number of responses is reached (doesn't loop infinitely)

In other words, rather than data accuracy, the purpose is to verify the **wiring of multi-agent control, context sharing, and termination conditions**.

## Utility Code
- `run_stream_pretty` — Helper for real-time display of streaming output
- `visualize_workflow` — Workflow SVG visualization

### Why `intermediate_outputs=True` Is Required

`GroupChatBuilder` defaults to `intermediate_outputs=False`.
In this case, `output_executors` is limited to only the orchestrator,
and participant agents' `AgentResponseUpdate` (token chunks) are
filtered by `Workflow._should_yield_output_event()` and
do not reach the consumer (`async for event in workflow.run(stream=True)`).

**Result**: Although streaming `print` should work, nothing is displayed,
and only `list[Message]` (final conversation log) is returned at the end.

Specifying `intermediate_outputs=True` sets `output_executors=None`,
which causes output events from all executors to be yielded,
allowing per-token streaming from participants to work correctly.

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "RoundRobinGroupChat",
    print_final: bool = True,
) -> list[Message]:
    """
    
    Executes workflow.run(task, stream=True) and displays streaming output
    in real-time within a Jupyter Notebook.

    Prerequisites (when using GroupChatBuilder):
      Build with GroupChatBuilder(..., intermediate_outputs=True).
      With the default (False), only the orchestrator's output is yielded,
      and participant AgentResponseUpdate is filtered out.

    Display:
      - AgentResponseUpdate → Tokens displayed incrementally (sys.stdout.write + flush)
      - On executor switch → Newline + name header
      - Final list[Message] → Displayed as CONVERSATION LOG summary

    Note:
      Uses start_as_current_span() to properly nest internal workflow spans
      as children of this span.
      _process_events is a regular async function, not an async generator,
      so no GeneratorExit issues occur when combined with context managers.
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Write with reliable flushing even in Jupyter"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- Executor switch notification -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"\U0001f916 {eid}: ")
                    last_executor_id = eid

            # ----- Output event -----
            elif event.type == "output":
                data = event.data

                # (1) Streaming chunk: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"\U0001f916 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) Non-streaming response: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"\U0001f916 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) Final conversation log: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # Use start_as_current_span() to nest internal workflow spans as children.
    #   _process_events is a regular async function so no context conflicts occur.
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # Trailing newline after stream
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    Function to output workflow graph information, save as SVG, and preview
    
    Args:
        workflow: The workflow object to visualize
        filename: The filename to save (without extension)
        show_preview: Whether to show the preview
    
    Returns:
        Path to the saved SVG file
    """
    # Create WorkflowViz object
    viz = WorkflowViz(workflow)
    
    # 3. Save as SVG file
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"\u2705 SVG file saved: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. Preview SVG
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("\u274c Error: 'graphviz' package is not installed")
        print("Installation: pip install agent-framework[viz] --pre")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"\u274c An error occurred: {e}")
        return None

In [ ]:
# Visualize workflow using the function
svg_path = visualize_workflow(
    workflow, 
    filename="collaborative_multi_agent_round_robin_workflow",
    show_preview=True
)

In [ ]:
task = "Look up the product details for Product ID: 1001, and write a description that includes inventory status and recent Twitter reputation."

# Execute run_stream with helper (stream display + final conversation collection)
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer)

In [ ]:
for msg in final_conversation:
    print(msg.to_dict())

## Scenario 1: Tokyo vs. Rural Areas Debate: Where Is the Richer Life?

Eight agents with diverse perspectives discuss the theme of "Tokyo or rural areas." A moderator strategically selects each speaker, creating a natural flow of dialogue.

## Agent Definitions (Debate Participants)

In [ ]:
# Moderator: Selects speakers and facilitates the discussion
moderator = Agent(
    name="Moderator",
    description="Selects the next speaker and guides the discussion",
    instructions="""You are a thoughtful moderator guiding a discussion on the theme: "Tokyo or rural areas — where does one live a richer life?"

The participants have diverse perspectives and experiences. Select speakers strategically:
- Constructively pit opposing opinions against each other
- Ensure all viewpoints are heard fairly
- Encourage arguments based on specific experiences and data
- Draw out logical reasoning, not just emotional arguments
- Discover realistic solutions and new perspectives

Select speakers who can:
1. Respond specifically to the immediately preceding statement
2. Reframe the issue from a different angle
3. Present persuasive opinions based on real-world experience
4. Pose essential questions that deepen the discussion

Terminate when:
- Multiple viewpoints have been sufficiently exchanged (6-8+ times)
- Major points have been discussed from multiple angles
- New insights or integrated perspectives have emerged

In the final_message, briefly summarize the key points and diverse values that emerged from the discussion.
""",
    client=chat_client,
)

# 8 participants with diverse perspectives
tokyo_salaryman = Agent(
    name="TokyoSalaryman",
    description="A corporate employee born and raised in Tokyo",
    instructions="""You are a salaried worker in your 30s, born and raised in Tokyo, working at a major corporation.
You value convenience, career opportunities, and cultural stimulation. It's hard for you to imagine life in rural areas.

Speak honestly. Feel free to:
- Specifically state the advantages of living in Tokyo
- Frankly express doubts and concerns about rural living
- Counter or question other participants' opinions
- Be concise (2-4 sentences), but persuasive
""",
    client=chat_client,
)

local_farmer = Agent(
    name="LocalFarmer",
    description="A farmer in their 40s working in a rural area",
    instructions="""You are in your 40s and have inherited the family farm that has been passed down for generations in a rural area.
You value nature, community, and a relaxed lifestyle. You can't understand the busyness of city life.

Speak honestly. Feel free to:
- Specifically describe the richness of rural living
- Frankly express doubts about living in Tokyo
- Respond to other participants' opinions from a rural perspective
- Be concise (2-4 sentences), but with genuine feeling
""",
    client=chat_client,
)

remote_worker = Agent(
    name="RemoteWorker",
    description="A remote worker living in a rural area",
    instructions="""You are in your 30s, working remotely for a Tokyo IT company while living in a rural city.
You practice a "best of both worlds" work style. You know the advantages and challenges of both.

Speak honestly. Feel free to:
- Talk about the reality of rural remote work (both positives and negatives)
- Propose a hybrid lifestyle between Tokyo and rural areas
- Respond to others' opinions based on real experience
- Be concise (2-4 sentences), but realistic
""",
    client=chat_client,
)

u_turn_entrepreneur = Agent(
    name="UTurnEntrepreneur",
    description="A former Tokyo employee who returned to their hometown to start a business",
    instructions="""You are in your late 30s and started a business in your hometown after working in Tokyo for 10 years.
Having experienced both, you can see differences that others can't. You believe in the potential of rural areas.

Speak honestly. Feel free to:
- Share perspectives from having experienced both Tokyo and rural life
- Frankly describe the reasons for your U-turn decision and the reality
- Share the possibilities and challenges of doing business in rural areas
- Be concise (2-4 sentences), but persuasive
""",
    client=chat_client,
)

college_student = Agent(
    name="CollegeStudent",
    description="A student from a rural area attending university in Tokyo",
    instructions="""You are in your early 20s, originally from a rural area, now attending university in Tokyo.
You're torn between returning home or getting a job in Tokyo. You carry the struggles of the younger generation.

Speak honestly. Feel free to:
- Talk about the appeal and anxieties of both Tokyo and rural areas from a young person's perspective
- Frankly express your uncertainties and hopes for the future
- Ask questions and react to opinions from older generations
- Be concise (2-4 sentences), but sincere
""",
    client=chat_client,
)

local_civil_servant = Agent(
    name="LocalCivilServant",
    description="A local government employee working on regional revitalization",
    instructions="""You are a civil servant in your 30s working on regional revitalization for a local government.
You witness the reality of population decline and Tokyo-centric concentration firsthand. You seriously consider the future of rural areas.

Speak honestly. Feel free to:
- Specifically describe the realistic challenges facing rural areas
- Point out problems of Tokyo concentration from a policy perspective
- Contribute to the discussion with data and case studies
- Be concise (2-4 sentences), but realistic
""",
    client=chat_client,
)

tokyo_real_estate_owner = Agent(
    name="TokyoRealEstateOwner",
    description="A real estate owner operating rental properties in central Tokyo",
    instructions="""You are a property owner in your 50s who owns multiple rental properties in central Tokyo.
You feel the Tokyo real estate market and the flow of people firsthand. You emphasize the economic perspective.

Speak honestly. Feel free to:
- Specifically discuss Tokyo's economic advantages
- Argue the value of location from a real estate investment perspective
- Express opinions from a market principles standpoint
- Be concise (2-4 sentences), but persuasive
""",
    client=chat_client,
)

dual_location = Agent(
    name="DualLocation",
    description="A person practicing dual-location living between Tokyo and a rural area",
    instructions="""You are in your 40s with an apartment in Tokyo and a renovated traditional house in a rural area, practicing dual-location living.
You can present the option of "both" rather than "either/or." You're exploring a new lifestyle.

Speak honestly. Feel free to:
- Specifically describe the reality of dual-location living (cost, time, benefits)
- Propose a third option beyond the binary choice
- Share your experience as a practitioner
- Be concise (2-4 sentences), but realistic
""",
    client=chat_client,
)

## Workflow Construction and Topic Setting

In [ ]:
# Build the debate workflow
debate_workflow = (
    GroupChatBuilder(
        participants=[tokyo_salaryman, local_farmer, remote_worker, u_turn_entrepreneur, 
                      college_student, local_civil_servant, tokyo_real_estate_owner, dual_location],
        orchestrator_agent=moderator,
        # Termination condition: 10 or more statements (each agent can speak multiple times)
        termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 10,
        # Required to display participant streaming output in real-time
        intermediate_outputs=True,
    )
    .build()
 )

# Debate topic
debate_topic = "Tokyo or rural areas — where do you think one lives a richer life?"

### What This Workflow Example Verifies (Scenario 1: Debate)

This example is a demo to verify that "the moderator (orchestrator) can drive the conversation" in a situation with many participants. It mainly checks the following.

- **Speaker selection from many participants**: The Moderator can continuously select the next speaker from 8 participants based on the flow of conversation
- **Accumulation of multiple perspectives under shared context**: Each participant's statement accumulates in history, and the next speaker can react based on it
- **Termination condition design**: Even in conversations with many participants, convergence can be achieved with criteria like `with_termination_condition(... >= 10)`
- **Designing for "conclusion summary"**: The Moderator's `final_message` instruction gives it the role of summarizing the discussion points

In [ ]:
# Visualize the workflow
svg_path = visualize_workflow(
    debate_workflow, 
    filename="philosophical_debate_workflow",
    show_preview=True
)

## Running the Debate

In [ ]:
# Run the philosophical debate
debate_conversation = await run_stream_pretty(
    debate_workflow, 
    debate_topic, 
    tracer=tracer,
    span_name="PhilosophicalDebate"
)

In [ ]:
for msg in debate_conversation:
    print(msg.to_dict())

## Scenario 2: Group Chat with a Simple Speaker Selector Function

This section demonstrates how to control speaker selection with a pure Python function using `with_orchestrator(selection_func=...)`.
The selection function receives `GroupChatState` and returns the name (string) of the participant to speak next.
Termination is controlled by `with_termination_condition(...)` or `with_max_rounds(...)`.

### Speaker Selector Function Definition

In [ ]:
def select_next_speaker(state: GroupChatState) -> str:
    """A simple speaker selector that alternates between the adder and multiplier.

    Args:
        state: GroupChatState (contains current_round / participants / conversation)

    Returns:
        Name of the next speaker (must be a name that exists in participants)
    """
    # current_round starts at 0 and increments each time a participant responds.
    # On even rounds (0,2,4,...) select Adder, on odd rounds (1,3,5,...) select Multiplier, alternating between the two.
    return "Adder" if (state.current_round % 2 == 0) else "Multiplier"

### Agent Definitions (Calculator)

In [ ]:
# Adder agent
adder = Agent(
    name="Adder",
    description="Adder (+100 operation)",
    instructions="""You are part of a calculation chain.

Important: Always check the conversation history and extract the number from the last message.

Your role:
1. Read the last message in the conversation history
2. Find the number (yen) in that message
3. Add 100 to that number
4. Show the calculation process and output the result

Output format: "[previous value] + 100 = [result] yen"

Examples:
- Previous message: "I have 100 yen" -> 100 + 100 = 200 yen
- Previous message: "200 x 2 = 400 yen" -> 400 + 100 = 500 yen
""",
    client=chat_client,
)

# Multiplier agent
multiplier = Agent(
    name="Multiplier",
    description="Multiplier (x2 operation)",
    instructions="""You are part of a calculation chain.

Important: Always check the conversation history and extract the number from the last message.

Your role:
1. Read the last message in the conversation history
2. Find the number (yen) in that message
3. Multiply that number by 2
4. Show the calculation process and output the result

Output format: "[previous value] x 2 = [result] yen"

Examples:
- Previous message: "100 + 100 = 200 yen" -> 200 x 2 = 400 yen
- Previous message: "400 + 100 = 500 yen" -> 500 x 2 = 1000 yen
""",
    client=chat_client,
)

### Workflow Construction (Calculator)

In [ ]:
# There are two ways to specify participants:
# 1. List format - uses agent.name attribute: participants=[adder, multiplier]
# 2. Dictionary format - explicit names: participants={"adder": adder, "multiplier": multiplier}
calculator_workflow = (
    GroupChatBuilder(
        participants=[adder, multiplier],  # Use agent.name as participant names
        selection_func=select_next_speaker,
        orchestrator_name="Calculator",
        # Required to display participant streaming output in real-time
        intermediate_outputs=True,
    )
    # Set max rounds as a safety net (prevent infinite loops)
    .with_max_rounds(4)
    # End after 4 participant responses (Adder x2 + Multiplier x2)
    .with_termination_condition(
        lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4
    )
    .build()
 )

In [ ]:
# Visualize the workflow
svg_path = visualize_workflow(
    calculator_workflow, 
    filename="calculator_workflow",
    show_preview=True
)

### Running the Calculator Workflow

In [ ]:
calculation_task = "I have 200 yen"

print(f"\n=== Calculation process: {calculation_task} ===\n")

# Execute the calculation
calculation_result = await run_stream_pretty(
    calculator_workflow, 
    calculation_task, 
    tracer=tracer,
    span_name="CalculatorWorkflow"
)

In [ ]:
for msg in calculation_result:
    print(msg.to_dict())

In [ ]:
for msg in calculation_result:
    print(msg.text)

The `CalculatorWorkflow` is not meant to verify "calculations themselves" but rather a "functional verification" to confirm that group chat orchestration works minimally. Specifically, it checks the following:

- **Whether speaker selection works as intended**: `selection_func` returns `"Adder"` / `"Multiplier"` alternately based on the parity of `state.current_round`, and speakers actually switch in that order
- **Whether context sharing works**: Subsequent agents can read the conversation history (previous message) and pick up numbers from it for the next calculation
- **Whether termination conditions work**: With `with_max_rounds(4)` and `with_termination_condition(... >= 4)`, the workflow stops at the expected number of rounds without looping infinitely
- **Streaming execution wiring verification**: `run_stream_pretty` correctly handles executor switching and final conversation log retrieval

## [Reference] Internal Logic of with_orchestrator
`with_orchestrator(...)` only configures "how to determine the next speaker," and there are **2 systems** for the speaker selection logic itself:

- **(A) When `selection_func=` is passed**  
  The Builder uses `GroupChatOrchestrator`, creates `GroupChatState(current_round, participants, conversation)` each turn, and calls `selection_func(state)`. If the return value is awaitable, it awaits it, **checks if the name exists in participants**, and makes that the next speaker.  

- **(B) When `agent=` is passed (LLM orchestrator selects)**  
  The Builder uses `AgentBasedGroupChatOrchestrator`, and each turn invokes the orchestrator agent (**Agent**) to **return who should speak next as JSON (structured output)**.  
  Internally, it embeds in the `instruction` string: "Return JSON containing `terminate/reason/next_speaker/final_message`" and "a list of valid participant names (case-sensitive)," adds it to the conversation, then executes `agent.run(... options={"response_format": AgentOrchestrationOutput})` → parses the returned JSON and adopts `next_speaker`.  
    - JSON (structured output) example:
        ```json
        {"terminate":false,"reason":"Researcher provided product details, inventory status, and Twitter mention data for product ID 1001. Next, Writer needs to create a comprehensive description incorporating this information.","next_speaker":"Writer","final_message":null}
        ```


Which path is taken depends on resolution at `build()` time: `agent=` results in `AgentBasedGroupChatOrchestrator`, while `selection_func=` results in `GroupChatOrchestrator`.  

Note: With the agent method, if `next_speaker` is an unregistered name, it can throw an exception at the send stage as "unregistered" (the participant check is not as explicit as in the selection_func method — it gets rejected at the send processing side).